# Simple MNIST convnet

**Author:** [fchollet](https://twitter.com/fchollet)<br>
**Date created:** 2015/06/19<br>
**Last modified:** 2020/04/21<br>
**Description:** A simple convnet that achieves ~99% test accuracy on MNIST.

## Setup

In [ ]:
import numpy as np
import keras
from keras import layers

## Prepare the data

In [ ]:
# Model / data parameters
num_classes = 10 #分类类别总数0-10一共十类
input_shape = (28, 28, 1) #表示单张输入图片的形状 / 维度(高度，宽度通道数)

# Load the data and split it between train and test sets
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# Scale images to the [0, 1] range 归一化，像素值取值一般为0-255，将其变为0-1之间的一个值
x_train = x_train.astype("float32") / 255
x_test = x_test.astype("float32") / 255
# Make sure images have shape (28, 28, 1) Keras/TensorFlow 的 Conv2D 层强制要求输入格式为 (样本数, 高, 宽, 通道数)
x_train = np.expand_dims(x_train, -1) #为数组新增一个维度
x_test = np.expand_dims(x_test, -1)
print("x_train shape:", x_train.shape)
print(x_train.shape[0], "train samples")
print(x_test.shape[0], "test samples")


# convert class vectors to binary class matrices 将类别向量转换为二进制矩阵，也就是独热编码。
y_train = keras.utils.to_categorical(y_train, num_classes) #原始标签 3 → 编码后：[0, 0, 0, 1, 0, 0, 0, 0, 0, 0]
y_test = keras.utils.to_categorical(y_test, num_classes) #整数标签 0~9 自带大小关系（比如 9>0），但手写数字分类中，类别之间没有大小、顺序之分。
###若直接用整数训练，模型会错误学习到 “数字 9 比数字 0 大” 这种无关特征；###
###独热编码把每个类别转为独立二进制位，类别之间相互平等，符合分类任务逻辑。###

## Build the model

In [ ]:
model = keras.Sequential(
    [
        keras.Input(shape=input_shape),
        layers.Conv2D(32, kernel_size=(3, 3), activation="relu"),#卷积核数量32 卷积核尺寸3*3 激活函数relu 输入：(28, 28, 1) 输出(26,26,32)
        layers.MaxPooling2D(pool_size=(2, 2)),#池化层，在 2×2 的窗口内，只保留最大值，丢弃其余数据 输入：(26, 26, 32) 输出(13,13,32)
        layers.Conv2D(64, kernel_size=(3, 3), activation="relu"),#输入 ：(13,13,32) 本层输出维度：(None, 11, 11, 64)
        layers.MaxPooling2D(pool_size=(2, 2)),#输入 (None, 11, 11, 64) 本层输出维度：(None, 5, 5, 64)
        layers.Flatten(), #展平成为一维数组
        layers.Dropout(0.5), #随机失活层训练阶段随机临时关闭 50% 的神经元
        layers.Dense(num_classes, activation="softmax"), #输出层，类别=10；激活函数softmax
    ]
)

model.summary()

## Train the model

In [ ]:
batch_size = 128 #一批128个样本
epochs = 15 #共学习15次

model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"]) #loss：损失值越大，代表模型预测越不准；训练的目标就是不断降低损失值。
#adam优化器 监督指标：准确率
model.fit(x_train, y_train, batch_size=batch_size, epochs=epochs, validation_split=0.1) #真正训练步骤，包含了两个超参数，从训练集中划分出10%验证集

## Evaluate the trained model

In [ ]:
score = model.evaluate(x_test, y_test, verbose=0)
print("Test loss:", score[0])
print("Test accuracy:", score[1])
#使用独立测试集评价泛化能力

## Relevant Chapters from Deep Learning with Python
- [Chapter 8: Image classification](https://deeplearningwithpython.io/chapters/chapter08_image-classification)
